# Setup


In [4]:
from optibook.synchronous_client import Exchange

import time
import logging

logger = logging.getLogger("client")
logger.setLevel("ERROR")

print("Setup was successful.")

Setup was successful.


# Connecting to the Exchange


In [ ]:
# create a connection to the exchange
exchange = Exchange()
_ = exchange.connect()

2025-11-01 13:55:56,155 [asyncio   ] [MainThread  ] Using selector: EpollSelector


2025-11-01 13:57:19,485 [client    ] [Thread-5 (_thread_entry_point)] Forcing a disconnect due to an error: Closing connection because someone else logged in with the same credentials. Only one session may be active at the same time.


# Getting market information


In [6]:
# Define a variable storing the instrument_id of the product we are interested in
instrument_id = "NVDA"

### Instrument Details


In [10]:
# Load all instruments (products) available on the exchange
instruments = exchange.get_instruments()
instruments

{'PFE_DUAL': Instrument(instrument_id=PFE_DUAL, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=1.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, instrument_group=PFE_DUAL, paused=False, expired=False),
 'SAN_DUAL': Instrument(instrument_id=SAN_DUAL, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=1.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, instrument_group=SAN_DUAL, paused=False, expired=False),
 'ING_DUAL': Instrument(instrument_id=ING_DUAL, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=1.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, instrument_group=ING_DUAL, paused=False, expired=False),
 'NVDA_DUAL': Instrument(instrument_id=NVDA_DUAL, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=1.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, instrument_group=NVDA_DUAL, paused=False, expired=False),
 'CSCO': Instrument(instr

In [11]:
# The returned type of exchange.get_instruments() is a dictionary, select a single instrument
instrument = instruments[instrument_id]
print(instrument)

Instrument(instrument_id=NVDA, tick_size=0.1, price_change_limit=PriceChangeLimit(absolute_change=1.0000, relative_change=10.00%), instrument_type=InstrumentType.STOCK, instrument_group=NVDA, paused=False, expired=False)


In [13]:
# From any instruments we can get a lot of data:
print(instrument.instrument_id)
print(instrument.instrument_type)
print(instrument.tick_size)

# For particular types of instruments, additional fields are populated (such as e.g. the option expiry, strike and kind, or the future expiry)
# For a STOCK those fields are equal to None
print(instrument.expiry)

print(instrument.option_kind)
print(instrument.strike)
print(instrument.base_instrument_id)

NVDA
InstrumentType.STOCK
0.1
None
None
None
None


In [14]:
# The InstrumentType is an Enum (enumerable) type, that means its value is one of a few supported categories.
# We can use this Enum to compare which specific type of Instrument we are dealing with.

# First import the InstrumentType enum (we would normally do this at the top of the file)
from optibook.common_types import InstrumentType

# Then make a comparison as such:
if instrument.instrument_type == InstrumentType.STOCK:
    print("It is a STOCK.")
elif instrument.instrument_type == InstrumentType.STOCK_OPTION:
    print("It is a STOCK_OPTION.")
elif instrument.instrument_type == InstrumentType.STOCK_FUTURE:
    print("It is a STOCK_FUTURE.")
else:
    print(f"It was another instrument_type: {instrument.instrument_type}.")

It is a STOCK.


### Order Book


In [28]:
# The exchange.get_last_price_book() method returns a large compound datatype storing bids and offers, with price and volumes, and a timestamp.
book = exchange.get_last_price_book(instrument_id)
book

PriceBook(CSCO 2025-11-01 11:43:30.255178)
 #bids | price | #asks 
       |  33.7 |  2500 
       |  33.6 |  480  
  2500 |  33.3 |       
  480  |  33.1 |       

In [29]:
# How do we take it apart? First, extract one side of the order book, the bids, say
bids = book.bids
bids

[PriceVolume(price=33.300000000000004, volume=2500),
 PriceVolume(price=33.1, volume=480)]

In [30]:
# That's a list of PriceVolume combinations, which makes sense, one line of the order book is always a price with corresponding volume and we have multiple such orders
# Let's find the most competitive order, which is always the first entry in the list
best_bid = bids[0]
best_bid

PriceVolume(price=33.300000000000004, volume=2500)

In [31]:
# That's still a compound of price and volume, we can simply select which one we want to see to finally end up with a numeric type
print(best_bid.price)
print(best_bid.volume)

33.300000000000004
2500


In [32]:
# Putting it all together we can also write this.
book.bids[0].price

33.300000000000004

In [33]:
# Note you will see an error on the above line if there are no bids at all available (the list will have no entry 0)
# Better is to check first
if book.bids:
    print(book.bids[0].price)
else:
    print("No bids in the order book at all.")

33.300000000000004


## Orders


### Inserting


In [34]:
# Insert bid LIMIT order - this trades against any current orders, and any remainders become new resting orders in the book
# The returned value is a 'InsertOrderResponse' which, if successfull, contains the order_id of the order
exchange.insert_order(instrument_id, price=10, volume=5, side="bid", order_type="limit")

InsertOrderResponse(success=True, order_id=1707646, error_reason='None')

In [35]:
# Insert ask LIMIT order - this trades against any current orders, and any remainders become new resting orders in the book
exchange.insert_order(instrument_id, price=20, volume=5, side="ask", order_type="limit")

InsertOrderResponse(success=True, order_id=1707648, error_reason='None')

In [36]:
# Insert bid IOC order - this order trades against any resting volume which has an equal or better price, but does not remain in the
# book if any volume is unfilled
exchange.insert_order(instrument_id, price=10, volume=5, side="bid", order_type="ioc")

InsertOrderResponse(success=True, order_id=1707650, error_reason='None')

In [37]:
# Insert ask IOC order - this order trades against any resting volume which has an equal or better price, but does not remain in the
# book if any volume is unfilled
exchange.insert_order(instrument_id, price=20, volume=5, side="ask", order_type="ioc")

InsertOrderResponse(success=True, order_id=1707659, error_reason='None')

### Deleting & amending


In [38]:
# Load a list of own outstanding/resting orders
exchange.get_outstanding_orders(instrument_id)

{1707646: OrderStatus(order_id=1707646, instrument_id=CSCO, price=10.0, volume=5, side=bid)}

In [39]:
# Delete an order by order_id, the returned DeleteOrderRequest contains a boolean showing whether the delete was succesful
exchange.delete_order(instrument_id, order_id=1234)

DeleteOrderResponse(success=False, error_reason='Could not find order id to delete')

In [40]:
# Delete all outstanding orders for an instrument
exchange.delete_orders(instrument_id)

In [41]:
# Change (amend) the volume of an existing resting order by order_id, the returned AmendOrderResponse contains a boolean showing whether the amend was succesful
exchange.amend_order(instrument_id, order_id=1234, volume=30)

AmendOrderResponse(success=False, error_reason='Could not find order id to amend')

### Public tradeticks


In [42]:
# Load and store a list of all public tradeticks since the instantiation of the Exchange (upto a max limit)
# These are trades between any participant, you may or may not have participated in these
trade_tick_history = exchange.get_trade_tick_history(instrument_id)

# Display last 5
trade_tick_history[-5:]

[TradeTick(timestamp=2025-11-01 11:43:28.726686, instrument_id=CSCO, price=33.2, volume=250, aggressor_side=bid, buyer=, seller=, trade_id=163545),
 TradeTick(timestamp=2025-11-01 11:43:29.759706, instrument_id=CSCO, price=33.4, volume=250, aggressor_side=bid, buyer=, seller=, trade_id=163553),
 TradeTick(timestamp=2025-11-01 11:43:30.988647, instrument_id=CSCO, price=33.300000000000004, volume=250, aggressor_side=ask, buyer=, seller=, trade_id=163562),
 TradeTick(timestamp=2025-11-01 11:43:31.435973, instrument_id=CSCO, price=33.300000000000004, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=163564),
 TradeTick(timestamp=2025-11-01 11:43:31.618980, instrument_id=CSCO, price=33.300000000000004, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=163567)]

In [43]:
# Poll all public tradeticks since the last time this method was called
exchange.poll_new_trade_ticks(instrument_id)

[TradeTick(timestamp=2025-11-01 11:43:27.580673, instrument_id=CSCO, price=33.2, volume=250, aggressor_side=bid, buyer=, seller=, trade_id=163541),
 TradeTick(timestamp=2025-11-01 11:43:28.726686, instrument_id=CSCO, price=33.2, volume=250, aggressor_side=bid, buyer=, seller=, trade_id=163545),
 TradeTick(timestamp=2025-11-01 11:43:29.759706, instrument_id=CSCO, price=33.4, volume=250, aggressor_side=bid, buyer=, seller=, trade_id=163553),
 TradeTick(timestamp=2025-11-01 11:43:30.988647, instrument_id=CSCO, price=33.300000000000004, volume=250, aggressor_side=ask, buyer=, seller=, trade_id=163562),
 TradeTick(timestamp=2025-11-01 11:43:31.435973, instrument_id=CSCO, price=33.300000000000004, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=163564),
 TradeTick(timestamp=2025-11-01 11:43:31.618980, instrument_id=CSCO, price=33.300000000000004, volume=5, aggressor_side=ask, buyer=, seller=, trade_id=163567)]

### Private trades


In [44]:
# Load and store a list of all private trades you participated in since the instantiation of the Exchange (upto a max limit)
trade_history = exchange.get_trade_history(instrument_id)

# Display last 5
trade_history[-5:]

[Trade(timestamp=2025-11-01 11:43:31.435973, order_id=1707648, trade_id=163564, instrument_id=CSCO, price=33.300000000000004, volume=5, side=ask),
 Trade(timestamp=2025-11-01 11:43:31.618980, order_id=1707659, trade_id=163567, instrument_id=CSCO, price=33.300000000000004, volume=5, side=ask)]

In [45]:
# Poll all private trades since the last time this method was called
exchange.poll_new_trades(instrument_id)

[Trade(timestamp=2025-11-01 11:43:31.435973, order_id=1707648, trade_id=163564, instrument_id=CSCO, price=33.300000000000004, volume=5, side=ask),
 Trade(timestamp=2025-11-01 11:43:31.618980, order_id=1707659, trade_id=163567, instrument_id=CSCO, price=33.300000000000004, volume=5, side=ask)]

### Position & PnL


In [46]:
# Load current positions in all instruments
exchange.get_positions()

{'NVDA': 0,
 'ING': 0,
 'SAN': 0,
 'CSCO_DUAL': 0,
 'PFE': 0,
 'CSCO': -10,
 'NVDA_DUAL': 0,
 'ING_DUAL': 0,
 'SAN_DUAL': 0,
 'PFE_DUAL': 0}

In [47]:
# Load current positions in all instruments, including cash component (how much did we invest or gain by trading each instrument)
exchange.get_positions_and_cash()

{'NVDA': {'volume': 0, 'cash': 0.0},
 'ING': {'volume': 0, 'cash': 0.0},
 'SAN': {'volume': 0, 'cash': 0.0},
 'CSCO_DUAL': {'volume': 0, 'cash': 0.0},
 'PFE': {'volume': 0, 'cash': 0.0},
 'CSCO': {'volume': -10, 'cash': 333.00000000000006},
 'NVDA_DUAL': {'volume': 0, 'cash': 0.0},
 'ING_DUAL': {'volume': 0, 'cash': 0.0},
 'SAN_DUAL': {'volume': 0, 'cash': 0.0},
 'PFE_DUAL': {'volume': 0, 'cash': 0.0}}

In [48]:
# Current PnL: sum of all cash components and value of the positions (evaluated against last-traded price)
exchange.get_pnl()

0.0

# Code Snippets


Combining a few of the exchange interactions above, we can write a code snippet to aggressively trade out of all currently held positions using IOC orders.

That means selling all products you are long, and buying back all you are short.

This is useful for starting with a clean slate, but doing so will come at a potentially large cost.

You are not guaranteed to end up with a zero position afterwards, only that volume which is also available to trade in the market will be bought/sold.


In [49]:
MIN_SELLING_PRICE = 0.10
MAX_BUYING_PRICE = 100000.00

positions = exchange.get_positions()
pnl = exchange.get_pnl()

print(f"Positions before: {positions}")
print(f"\nPnL before: {pnl:.2f}")

print(f"\nTrading out of positions")
for iid, pos in positions.items():
    if pos > 0:
        print(
            f"-- Inserting sell order for {pos} lots of {iid}, with limit price {MIN_SELLING_PRICE:.2f}"
        )
        exchange.insert_order(
            iid, price=MIN_SELLING_PRICE, volume=pos, side="ask", order_type="ioc"
        )
    elif pos < 0:
        print(
            f"-- Inserting buy order for {abs(pos)} lots of {iid}, with limit price {MAX_BUYING_PRICE:.2f}"
        )
        exchange.insert_order(
            iid, price=MAX_BUYING_PRICE, volume=-pos, side="bid", order_type="ioc"
        )
    else:
        print(f"-- No initial position in {iid}, skipping..")

    time.sleep(0.10)

time.sleep(1.0)

positions = exchange.get_positions()
pnl = exchange.get_pnl()
print(f"\nPositions after: {positions}")
print(f"\nPnL after: {pnl:.2f}")

Positions before: {'NVDA': 0, 'ING': 0, 'SAN': 0, 'CSCO_DUAL': 0, 'PFE': 0, 'CSCO': -10, 'NVDA_DUAL': 0, 'ING_DUAL': 0, 'SAN_DUAL': 0, 'PFE_DUAL': 0}

PnL before: 0.00

Trading out of positions
-- No initial position in NVDA, skipping..
-- No initial position in ING, skipping..
-- No initial position in SAN, skipping..
-- No initial position in CSCO_DUAL, skipping..
-- No initial position in PFE, skipping..
-- Inserting buy order for 10 lots of CSCO, with limit price 100000.00
-- No initial position in NVDA_DUAL, skipping..
-- No initial position in ING_DUAL, skipping..
-- No initial position in SAN_DUAL, skipping..
-- No initial position in PFE_DUAL, skipping..

Positions after: {'NVDA': 0, 'ING': 0, 'SAN': 0, 'CSCO_DUAL': 0, 'PFE': 0, 'CSCO': 0, 'NVDA_DUAL': 0, 'ING_DUAL': 0, 'SAN_DUAL': 0, 'PFE_DUAL': 0}

PnL after: -3.00
